# Fase 1 — V1 / V2 / V3 / control (Qwen3-0.6B, protocolo del HANDOVER)

**Cómo correrlo en Kaggle (sin dejar el computador prendido):**
1. *File → Import Notebook* y sube este `.ipynb`.
2. Panel derecho → *Settings*: **Accelerator = GPU T4 x2**, **Internet = On** (requiere teléfono verificado).
3. Arriba a la derecha: **Save Version → "Save & Run All (Commit)"**. Ya puedes cerrar el navegador: corre en segundo plano.
4. Al terminar, en la versión guardada → pestaña *Output*: `out/phase1_seed0.json`, `out/phase1_seed1.json`,
   `out/v1_screening.json`, `tables/tables.md` y `logs/`.

**Qué hace:** preflight rápido (falla en minutos si algo está mal, antes de gastar horas) → verifica la densa
(wt2 ≈ 27.15, Pile ≈ 9.38; aborta si se desvía >3 %) → elige (k, bz) de las escalas log con calibración →
corre las semillas 0 y 1 en paralelo (una por GPU): Hessianos, rotación Hadamard, GPTQ con todas las opciones,
sensibilidad, mochila en 2.04…4.25 bpw, y evalúa cada modelo (PPL, KL, top-1, lm-eval 0-shot ×1000, flips, set dorado).

**Tiempo estimado con T4 x2:** ~4–5 h (no supera el límite de 12 h por sesión). Con una sola GPU las semillas corren
en serie (~8–9 h): en ese caso pon `SEEDS = [0]` y corre la semilla 1 en otra sesión.
Si algo se corta, los JSON guardan cada modelo terminado: adjunta la salida anterior como *Input* y pon su ruta en
`RESUME_DIR` para continuar donde quedó.

In [ ]:
SEEDS = [0, 1]          # rotaciones aleatorias (mínimo 2 según el HANDOVER)
RESUME_DIR = None       # p.ej. '/kaggle/input/<salida-anterior>/out' para continuar
LMEVAL_LIMIT = 1000

In [ ]:
!pip install -q transformers==5.19.0 lm-eval==0.4.13 rouge-score==0.1.2 gguf==0.19.0 sentencepiece

In [ ]:
import os
os.makedirs('/kaggle/working/qb/qbench', exist_ok=True)
%cd /kaggle/working/qb

In [ ]:
%%writefile qbench/__init__.py
"""Quantization benchmark (HANDOVER protocol)."""

In [ ]:
%%writefile qbench/protocol.py
"""HANDOVER §2: data, calibration and evaluation protocol (identical for every method)."""
import hashlib
import os

import numpy as np
import pandas as pd
import torch

MP = "Qwen/Qwen3-0.6B"
SEQ = 512
EVALTOK = 20000
DENSE_REF = {"wt2": 27.15, "pile": 9.38}
LINEAR_SUFFIXES = [
    "self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj", "self_attn.o_proj",
    "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj",
]


def linear_names(n_layers):
    return [f"model.layers.{i}.{s}" for i in range(n_layers) for s in LINEAR_SUFFIXES]


def download_raw():
    from huggingface_hub import hf_hub_download
    pq = hf_hub_download("NeelNanda/pile-10k", "data/train-00000-of-00001-4746b8785c874cc7.parquet",
                         repo_type="dataset")
    docs = [t for t in pd.read_parquet(pq)["text"].tolist() if isinstance(t, str) and t.strip()]
    wq = hf_hub_download("Salesforce/wikitext", "wikitext-2-raw-v1/test-00000-of-00001.parquet",
                         repo_type="dataset")
    wt = [x for x in pd.read_parquet(wq)["text"].tolist() if isinstance(x, str) and x.strip()]
    return docs, wt


def build_protocol(tok, docs, wt):
    """Line-by-line transcription of HANDOVER §2 (same RNG calls in the same order)."""
    docs = list(docs)
    np.random.RandomState(0).shuffle(docs)
    pile_test = tok("\n\n".join(docs[:200]), return_tensors="pt").input_ids[0][:EVALTOK]
    sep = tok("\n\n").input_ids
    pile_train = torch.tensor([x for ids in tok(docs[200:]).input_ids for x in ids + sep], dtype=torch.long)
    wt_test = tok("\n\n".join(wt), return_tensors="pt").input_ids[0][:EVALTOK]

    def batch(n, L=SEQ):
        js = torch.randint(0, len(pile_train) - L - 1, (n,))
        return torch.stack([pile_train[j:j + L + 1] for j in js])

    torch.manual_seed(1234)
    calib = batch(32)[:, :-1]
    sens_ids = batch(8, 512).reshape(-1)
    calib_text = "\n".join(tok.decode(c) for c in calib)
    return dict(pile_test=pile_test, wt_test=wt_test, calib=calib, sens_ids=sens_ids, calib_text=calib_text)


def fingerprint(P):
    h = {}
    for k in ("pile_test", "wt_test", "calib", "sens_ids"):
        h[k] = hashlib.sha256(P[k].numpy().tobytes()).hexdigest()[:16]
    h["calib_text"] = hashlib.sha256(P["calib_text"].encode()).hexdigest()[:16]
    h["n_tokens"] = {k: int(P[k].numel()) for k in ("pile_test", "wt_test", "calib", "sens_ids")}
    return h


def load_protocol(tok, cache_path=None, raw=None):
    if cache_path and os.path.exists(cache_path):
        return torch.load(cache_path)
    docs, wt = raw if raw is not None else download_raw()
    P = build_protocol(tok, docs, wt)
    if cache_path:
        torch.save(P, cache_path)
    return P


def load_model(path, device, dtype=torch.float32):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForCausalLM.from_pretrained(path, dtype=dtype, attn_implementation="sdpa")
    model = model.to(device).eval()
    model.requires_grad_(False)
    return tok, model


def model_snapshot(model_id=MP):
    if os.path.isdir(model_id):
        return model_id
    from huggingface_hub import snapshot_download
    return snapshot_download(model_id, allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.jinja"])

In [ ]:
%%writefile qbench/hadamard.py
"""HANDOVER §4.2: randomized Hadamard rotation (input side, one seed per matrix)."""
import functools
import math

import torch


def sylvester(n):
    H = torch.ones(1, 1, dtype=torch.float64)
    while H.shape[0] < n:
        H = torch.cat([torch.cat([H, H], 1), torch.cat([H, -H], 1)], 0)
    return H


def is_prime(q):
    return q > 1 and all(q % d for d in range(2, int(q ** .5) + 1))


def paley1(q):
    qr = {(x * x) % q for x in range(1, q)}
    chi = lambda a: 0 if a % q == 0 else (1 if a % q in qr else -1)
    Q = torch.tensor([[chi(j - i) for j in range(q)] for i in range(q)], dtype=torch.float64)
    S = torch.zeros(q + 1, q + 1, dtype=torch.float64)
    S[0, 1:] = 1
    S[1:, 0] = -1
    S[1:, 1:] = Q
    return S + torch.eye(q + 1, dtype=torch.float64)


@functools.lru_cache(maxsize=None)
def hadamard(n):
    for m in sorted({1, 2} | {q + 1 for q in range(3, n) if is_prime(q) and q % 4 == 3 and n % (q + 1) == 0}):
        k = n // m
        if n % m == 0 and (k & (k - 1)) == 0:
            Hm = torch.ones(1, 1, dtype=torch.float64) if m == 1 else (sylvester(2) if m == 2 else paley1(m - 1))
            return torch.kron(sylvester(k), Hm) / math.sqrt(n)
    raise ValueError(f"no Hadamard construction for n={n}")


def rotation_signs(n, seed, matrix_index):
    g = torch.Generator().manual_seed(1_000_003 * (seed + 1) + matrix_index)
    return torch.randint(0, 2, (n,), generator=g).double() * 2 - 1


def had_rot(n, signs):
    """V = H_n · diag(s) (H_n already normalized by 1/sqrt(n)); V is orthogonal."""
    return hadamard(n) * signs[None, :]

In [ ]:
%%writefile qbench/quant.py
"""HANDOVER §4.3–4.7 (standardized grid, GPTQ, PT2-style ternary, log scales) and §5 (bit accounting)."""
import math
from dataclasses import dataclass

import numpy as np
import torch

from .hadamard import hadamard

CS = [float(c) for c in np.linspace(1.0, 5.0, 33)]


@dataclass(frozen=True)
class Opt:
    kind: str            # "q" (asymmetric grid) | "tern"
    bits: int = 0        # grid bits (unused for "tern")
    scale: str = "fp16"  # "fp16" | "log"
    k: int = 0           # log2-scale index bits (scale == "log")
    bz: int = 0          # zero-point / mu bits (scale == "log")
    group: int = 128

    @property
    def name(self):
        base = "tern" if self.kind == "tern" else f"q{self.bits}"
        if self.scale == "fp16" and self.group == 128:
            return base
        return f"{base}.{self.scale}.k{self.k}z{self.bz}g{self.group}"


def opt_bits(opt, rows, cols):
    """§5: payload + per-(row, group) metadata + per-tensor log range (2×fp32) + Hadamard signs (cols bits)."""
    ng = cols // opt.group
    payload = rows * cols * opt.bits if opt.kind == "q" else math.ceil(rows * cols / 5) * 8
    meta = rows * ng * (32 if opt.scale == "fp16" else opt.k + opt.bz)
    tensor = 64 if opt.scale == "log" else 0
    return payload + meta + tensor + cols


# ---------------------------------------------------------------- §4.3 standardized grid (vectorized over c)
def grid_clip(blk, qmax):
    mu = blk.mean(1, keepdim=True)
    sd = blk.std(1, keepdim=True) + 1e-12
    c = torch.tensor(CS, dtype=blk.dtype, device=blk.device).view(1, -1, 1)
    l = mu.unsqueeze(1) - c * sd.unsqueeze(1)
    s = 2 * c * sd.unsqueeze(1) / qmax
    x = blk.unsqueeze(1)
    e = ((torch.clamp(torch.round((x - l) / s), 0, qmax) * s + l - x) ** 2).sum(-1)
    cj = c.view(-1)[e.argmin(1)].unsqueeze(1)          # argmin keeps the first minimum, like the spec's strict '<'
    lo, hi = mu - cj * sd, mu + cj * sd
    s = (hi - lo) / qmax
    return lo, torch.where(s == 0, torch.ones_like(s), s)


def grid_err(blk, lo, s, qmax):
    """Squared error per (row, candidate). blk (r,G); lo,s (r,C)."""
    x = blk.unsqueeze(1)
    l, ss = lo.unsqueeze(-1), s.unsqueeze(-1)
    return ((torch.clamp(torch.round((x - l) / ss), 0, qmax) * ss + l - x) ** 2).sum(-1)


# ---------------------------------------------------------------- §4.5 ternary (ITF + AGA), verbatim
def itf(Wb, iters=10):
    mu = Wb.mean(1, keepdim=True)
    Wt = Wb - mu
    D = 0.75 * Wt.abs().mean(1, keepdim=True)
    T = torch.sign(Wt) * (Wt.abs() > D)
    al = (T * Wt).sum(1, keepdim=True) / T.abs().sum(1, keepdim=True).clamp(min=1)
    g = Wb.shape[1]
    Tp = None
    for _ in range(iters):
        a = (T * T).sum(1, keepdim=True); s = T.sum(1, keepdim=True)
        wt = (Wb * T).sum(1, keepdim=True); w1 = Wb.sum(1, keepdim=True)
        den = g * a - s * s; ok = den.abs() > 1e-9; dd = torch.where(ok, den, torch.ones_like(den))
        al = torch.where(ok, (g * wt - s * w1) / dd, al); mu = torch.where(ok, (a * w1 - s * wt) / dd, mu)
        al = torch.where(al.abs() < 1e-12, torch.full_like(al, 1e-6), al)
        T = torch.clamp(torch.round((Wb - mu) / al), -1, 1)
        if Tp is not None and torch.equal(T, Tp):
            break
        Tp = T.clone()
    return al, mu, T


def aga(Wb, T, S):
    one = torch.ones_like(T); TS = T @ S; OS = one @ S; WS = Wb @ S
    tSt = (TS * T).sum(1, keepdim=True); tS1 = (TS * one).sum(1, keepdim=True); oSo = (OS * one).sum(1, keepdim=True)
    wSt = (WS * T).sum(1, keepdim=True); wS1 = (WS * one).sum(1, keepdim=True)
    den = tSt * oSo - tS1 * tS1; ok = den.abs() > 1e-12; dd = torch.where(ok, den, torch.ones_like(den))
    return (wSt * oSo - tS1 * wS1) / dd, (tSt * wS1 - tS1 * wSt) / dd, ok


def tgrid(Wb, S):
    al, mu, T = itf(Wb)
    a2, m2, ok = aga(Wb, T, S)
    al = torch.where(ok, a2, al); mu = torch.where(ok, m2, mu)
    return torch.where(al.abs() < 1e-12, torch.full_like(al, 1e-6), al), mu


def tern_err(blk, mu, al, S):
    """Hessian-weighted error per (row, candidate), the same objective AGA minimizes. mu, al (r,C)."""
    x = blk.unsqueeze(1)
    m, a = mu.unsqueeze(-1), al.unsqueeze(-1)
    R = x - (m + a * torch.clamp(torch.round((x - m) / a), -1, 1))
    return torch.einsum("rcj,jk,rck->rc", R, S, R)


# ---------------------------------------------------------------- grid finders (called at each group start)
NB3 = (-1.0, 0.0, 1.0)


class FinderQ:
    """fp16 (lo, s) per row and group."""

    def __init__(self, opt):
        self.qmax = 2 ** opt.bits - 1

    def __call__(self, blk, S):
        lo, s = grid_clip(blk, self.qmax)
        lo, s = lo.half().float(), s.half().float()
        return lo, torch.where(s == 0, torch.ones_like(s), s)


class FinderQLog:
    """§4.7: s on a per-tensor log2 grid (k bits), lo = -u·ŝ with u on a bz-bit grid over [-0.5, qmax+0.5].
    Only already-quantized (ŝ, lo) pairs are evaluated: {l0-1, l0, l0+1} × {z0-1, z0, z0+1}, min squared error."""

    def __init__(self, opt, lo_t, step):
        self.qmax = 2 ** opt.bits - 1
        self.L, self.Z = 2 ** opt.k - 1, 2 ** opt.bz - 1
        self.r = (self.qmax + 1) / 2 ** opt.bz
        self.lo_t, self.step = lo_t, step

    def decode(self, l, z):
        sh = torch.exp2(self.lo_t + l * self.step)
        return -sh * (-0.5 + (z + 0.5) * self.r), sh

    def __call__(self, blk, S):
        lo_c, s_c = grid_clip(blk, self.qmax)
        nb = torch.tensor(NB3, device=blk.device)
        l0 = torch.round((torch.log2(s_c) - self.lo_t) / self.step)
        ls = (l0 + nb).clamp(0, self.L)                                    # (r,3)
        sh = torch.exp2(self.lo_t + ls * self.step)
        z0 = torch.round((-lo_c / sh + 0.5) / self.r - 0.5)
        zs = (z0.unsqueeze(-1) + nb).clamp(0, self.Z)                       # (r,3,3)
        lo, s = self.decode(ls.unsqueeze(-1).expand_as(zs), zs)
        lo, s = lo.reshape(len(blk), 9), s.reshape(len(blk), 9)
        j = grid_err(blk, lo, s, self.qmax).argmin(1, keepdim=True)
        return lo.gather(1, j), s.gather(1, j)


class FinderTern:
    """fp16 (mu, |alpha|) per row and group (the ternary grid {mu-a, mu, mu+a} is symmetric in the sign of a)."""

    def __init__(self, opt):
        pass

    def __call__(self, blk, S):
        al, mu = tgrid(blk, S)
        al = al.abs().half().float()
        return mu.half().float(), torch.where(al == 0, torch.full_like(al, 1e-6), al)


class FinderTernLog:
    """V3: |alpha| on a per-tensor log2 grid (k bits), mu = alpha·(-1 + 2z/Z) with z on bz bits;
    3×3 quantized candidates, chosen by the AGA (Hessian-weighted) error."""

    def __init__(self, opt, lo_t, step):
        self.L, self.Z = 2 ** opt.k - 1, 2 ** opt.bz - 1
        self.lo_t, self.step = lo_t, step

    def __call__(self, blk, S):
        al_c, mu_c = tgrid(blk, S)
        nb = torch.tensor(NB3, device=blk.device)
        l0 = torch.round((torch.log2(al_c.abs()) - self.lo_t) / self.step)
        ls = (l0 + nb).clamp(0, self.L)
        ah = torch.exp2(self.lo_t + ls * self.step)                         # (r,3)
        z0 = torch.round((mu_c / ah + 1) / 2 * self.Z)
        zs = (z0.unsqueeze(-1) + nb).clamp(0, self.Z)                       # (r,3,3)
        a = ah.unsqueeze(-1).expand_as(zs)
        mu = a * (-1 + 2 * zs / self.Z)
        mu, a = mu.reshape(len(blk), 9), a.reshape(len(blk), 9)
        j = tern_err(blk, mu, a, S).argmin(1, keepdim=True)
        return mu.gather(1, j), a.gather(1, j)


# ---------------------------------------------------------------- per-tensor log ranges (pre-pass on W' before GPTQ)
def log_range(vals, k):
    v = vals.flatten().float()
    lo_t = torch.quantile(v, 0.01).item() - 0.5
    hi_t = v.max().item() + 0.5
    step = (hi_t - lo_t) / (2 ** k - 1)
    return float(np.float32(lo_t)), float(np.float32(step))


def prepass_q(Wr, opt):
    r, n = Wr.shape
    _, s = grid_clip(Wr.reshape(r * (n // opt.group), opt.group), 2 ** opt.bits - 1)
    return log_range(torch.log2(s), opt.k)


def prepass_tern(Wr, Hr, opt):
    G = opt.group
    als = [tgrid(Wr[:, j:j + G], Hr[j:j + G, j:j + G])[0].abs() for j in range(0, Wr.shape[1], G)]
    return log_range(torch.log2(torch.cat(als, 1)), opt.k)


def make_finder(opt, Wr, Hr):
    if opt.kind == "q":
        if opt.scale == "fp16":
            return FinderQ(opt), None
        lt = prepass_q(Wr, opt)
        return FinderQLog(opt, *lt), lt
    if opt.scale == "fp16":
        return FinderTern(opt), None
    lt = prepass_tern(Wr, Hr, opt)
    return FinderTernLog(opt, *lt), lt


# ---------------------------------------------------------------- §4.4 GPTQ (lazy-batch form of the spec's column loop)
def prep_hinv(Hraw):
    """prepH + Cholesky of the inverse, done in float64 for robustness (same math as the spec)."""
    n = Hraw.shape[0]
    H = Hraw.double() * 2
    dead = torch.diag(H) == 0
    d = dead.nonzero().flatten()
    H[d, d] = 1.0
    idx = torch.arange(n, device=H.device)
    H[idx, idx] += 0.01 * torch.diag(H).mean()
    L = torch.linalg.cholesky(H)
    Hinv = torch.linalg.cholesky(torch.cholesky_inverse(L), upper=True)
    return Hinv.float(), dead


@dataclass
class QMat:
    opt: Opt
    codes: torch.Tensor    # int8 (rows, cols): grid index in [0, qmax] or trit in {-1, 0, 1}
    p0: torch.Tensor       # float32 (rows, groups): lo (grid) | mu (ternary)
    p1: torch.Tensor       # float32 (rows, groups): s (grid)  | alpha (ternary)
    signs: torch.Tensor    # float64 (cols,): Hadamard signs
    tensor_params: tuple = None

    def dequant_rot(self, device):
        G = self.opt.group
        c = self.codes.to(device).float()
        p0 = self.p0.to(device).repeat_interleave(G, 1)
        p1 = self.p1.to(device).repeat_interleave(G, 1)
        return (p0 + p1 * c) if self.opt.kind == "tern" else (c * p1 + p0)

    def weff(self, device, dtype=torch.float32):
        """W_eff = Q · V with V = H_n · diag(s)."""
        Q = self.dequant_rot(device).to(dtype)
        n = Q.shape[1]
        return ((Q @ hadamard_on(n, device, dtype)) * self.signs.to(device, dtype)[None, :]).float()


_HCACHE = {}


def hadamard_on(n, device, dtype=torch.float64):
    key = (n, str(device), dtype)
    if key not in _HCACHE:
        _HCACHE[key] = hadamard(n).to(device, dtype)
    return _HCACHE[key]


def rotate(W, Hraw, signs):
    """W' = W·Vᵀ, H' = V·H·Vᵀ (float64)."""
    dev = W.device
    n = W.shape[1]
    Hn = hadamard_on(n, dev)
    s = signs.to(dev)
    Wr = (W.double() * s[None, :]) @ Hn.T
    Hr = Hn @ (s[:, None] * Hraw.to(dev).double() * s[None, :]) @ Hn.T
    return Wr, Hr


@torch.no_grad()
def gptq_quantize(Wr, Hr, Hinv, dead, opt, finder, blocksize=128):
    W = Wr.float().clone()
    W[:, dead] = 0
    Hrf = Hr.float()
    r, n = W.shape
    G = opt.group
    assert blocksize % G == 0 and n % G == 0
    tern = opt.kind == "tern"
    qmax = 2 ** opt.bits - 1
    codes = torch.empty((r, n), dtype=torch.int8, device=W.device)
    P0 = torch.empty((r, n // G), device=W.device)
    P1 = torch.empty_like(P0)
    for i1 in range(0, n, blocksize):
        i2 = min(i1 + blocksize, n)
        W1 = W[:, i1:i2].clone()
        Err1 = torch.zeros_like(W1)
        Hinv1 = Hinv[i1:i2, i1:i2]
        for i in range(i2 - i1):
            col = i1 + i
            if col % G == 0:
                p0, p1 = finder(W1[:, i:i + G], Hrf[col:col + G, col:col + G])
                P0[:, col // G] = p0[:, 0]
                P1[:, col // G] = p1[:, 0]
            w = W1[:, i]
            if tern:
                t = torch.clamp(torch.round((w[:, None] - p0) / p1), -1, 1)
                q = (p0 + p1 * t)[:, 0]
            else:
                t = torch.clamp(torch.round((w[:, None] - p0) / p1), 0, qmax)
                q = (t * p1 + p0)[:, 0]
            codes[:, col] = t[:, 0].to(torch.int8)
            err = (w - q) / Hinv1[i, i]
            W1[:, i] = q
            if i + 1 < i2 - i1:
                W1[:, i + 1:] -= err[:, None] * Hinv1[i, i + 1:][None, :]
            Err1[:, i] = err
        if i2 < n:
            W[:, i2:] -= Err1 @ Hinv[i1:i2, i2:]
    return codes, P0, P1


def quantize_matrix(Wr, Hr, Hinv, dead, opt, signs):
    finder, tp = make_finder(opt, Wr.float(), Hr.float())
    codes, P0, P1 = gptq_quantize(Wr, Hr, Hinv, dead, opt, finder)
    return QMat(opt, codes.cpu(), P0.cpu(), P1.cpu(), signs, tp)

In [ ]:
%%writefile qbench/hessian.py
"""HANDOVER §4.1: H = XᵀX / N per linear, from the ORIGINAL model on the 32 calibration windows."""
import torch

# Linears that read the same activation share one Hessian (q/k/v read input_layernorm, gate/up read
# post_attention_layernorm), so each distinct input is accumulated once.
SHARED = {"self_attn.k_proj": "self_attn.q_proj", "self_attn.v_proj": "self_attn.q_proj", "mlp.up_proj": "mlp.gate_proj"}


@torch.no_grad()
def collect_hessians(model, calib, names, device, bs=4):
    mods = dict(model.named_modules())
    owners = [n for n in names if not any(n.endswith(k) for k in SHARED)]
    acc, hooks = {}, []

    def mk(name):
        def hook(mod, inp, out):
            x = inp[0].detach().reshape(-1, inp[0].shape[-1]).float()
            h = (x.T @ x).double()
            acc[name] = h if name not in acc else acc[name] + h
        return hook

    for n in owners:
        hooks.append(mods[n].register_forward_hook(mk(n)))
    try:
        for i in range(0, len(calib), bs):
            model(calib[i:i + bs].to(device))
    finally:
        for h in hooks:
            h.remove()
    N = calib.numel()
    H = {n: (acc[n] / N).float().cpu() for n in owners}
    del acc
    for n in names:
        if n not in H:
            pre, suf = n.split(".", 3)[:3], n.split(".", 3)[3]
            H[n] = H[".".join(pre) + "." + SHARED[suf]]
    return H

In [ ]:
%%writefile qbench/mixed.py
"""HANDOVER §4.6: per-matrix sensitivity on calibration data + greedy knapsack."""
import time


def sensitivity(bench, store, names, opt_names, ref_opt, sens_ids, log=print):
    """Δ(m,o) = log ppl(sens_ids | only m in o, rest in ref_opt) − log ppl(sens_ids | all in ref_opt)."""
    bench.load({m: store[m][ref_opt].weff(bench.device) for m in names})
    base = bench.logppl(sens_ids)
    D = {m: {ref_opt: 0.0} for m in names}
    t0 = time.time()
    for j, m in enumerate(names):
        for o in opt_names:
            if o == ref_opt:
                continue
            bench.set_one(m, store[m][o].weff(bench.device))
            D[m][o] = bench.logppl(sens_ids) - base
        bench.set_one(m, store[m][ref_opt].weff(bench.device))
        if (j + 1) % 28 == 0:
            log(f"    sensitivity {j + 1}/{len(names)} ({time.time() - t0:.0f}s)")
    return base, D


def knapsack(names, opt_names, bits, delta, budget):
    """Start from the cheapest option per matrix; repeatedly apply the upgrade (m: o→o') with the largest
    (Δ(m,o) − Δ(m,o')) / (bits(o') − bits(o)) that keeps the total within budget. Only upgrades that lower Δ
    are considered. Returns None if even the cheapest assignment exceeds the budget (infeasible)."""
    choice = {m: min(opt_names, key=lambda o: bits[m][o]) for m in names}
    total = sum(bits[m][choice[m]] for m in names)
    if total > budget:
        return None, total
    while True:
        best = None
        for m in names:
            o = choice[m]
            for o2 in opt_names:
                db = bits[m][o2] - bits[m][o]
                gain = delta[m][o] - delta[m][o2]
                if db <= 0 or gain <= 0 or total + db > budget:
                    continue
                r = gain / db
                if best is None or r > best[0]:
                    best = (r, m, o2, db)
        if best is None:
            return choice, total
        _, m, o2, db = best
        choice[m] = o2
        total += db

In [ ]:
%%writefile qbench/bench.py
"""Shared evaluation bench (HANDOVER §3): a frozen dense fp32 model plus a working copy whose 196 block linears
are overwritten with each method's effective weights. Embeddings, norms and lm_head always stay dense."""
import copy
import json
import math
import time

import numpy as np
import torch
import torch.nn.functional as F

LMEVAL_TASKS = ["arc_easy", "arc_challenge", "hellaswag", "piqa", "winogrande"]


def windows(ids, win=512):
    out = []
    for i in range(0, len(ids) - 1, win):
        c = ids[i:i + win + 1]
        if len(c) < 2:
            break
        out.append(c)
    return out


def batched_windows(ids, bs, win=512):
    ws = windows(ids, win)
    full = [w for w in ws if len(w) == win + 1]
    rest = [w for w in ws if len(w) != win + 1]
    for i in range(0, len(full), bs):
        yield torch.stack(full[i:i + bs])
    for w in rest:
        yield w[None]


class Bench:
    def __init__(self, dense, tok, names, device, ppl_bs=4, kl_bs=2, lmeval_tasks=None, lmeval_include=None):
        self.dense, self.tok, self.names, self.device = dense, tok, names, device
        self.lmeval_tasks = list(lmeval_tasks or LMEVAL_TASKS)
        self.lmeval_include = lmeval_include
        self.work = copy.deepcopy(dense)
        self.work.requires_grad_(False)
        wm = dict(self.work.named_modules())
        dm = dict(self.dense.named_modules())
        self.mods = {n: wm[n] for n in names}
        self.orig = {n: dm[n].weight for n in names}
        self.ppl_bs, self.kl_bs = ppl_bs, kl_bs

    # ---------------------------------------------------------- weights
    def set_one(self, name, W):
        w = self.mods[name].weight
        assert W.shape == w.shape, (name, W.shape, w.shape)
        w.data.copy_(W.to(w.device, w.dtype))

    def load(self, weights):
        missing = set(self.names) - set(weights)
        assert not missing, f"missing {len(missing)} linears, e.g. {sorted(missing)[:3]}"
        for n, W in weights.items():
            self.set_one(n, W)

    def reset(self):
        for n in self.names:
            self.mods[n].weight.data.copy_(self.orig[n].data)

    # ---------------------------------------------------------- perplexity (same windows as the spec's ppl())
    @torch.no_grad()
    def nll(self, ids, model=None):
        model = model or self.work
        nll, nt = 0.0, 0
        for x in batched_windows(ids, self.ppl_bs):
            x = x.to(self.device)
            logits = model(x[:, :-1]).logits.float()
            nll += F.cross_entropy(logits.reshape(-1, logits.shape[-1]), x[:, 1:].reshape(-1),
                                   reduction="none").double().sum().item()
            nt += x.shape[0] * (x.shape[1] - 1)
            del logits
        return nll, nt

    def logppl(self, ids, model=None):
        nll, nt = self.nll(ids, model)
        return nll / nt

    def ppl(self, ids, model=None):
        return float(math.exp(self.logppl(ids, model)))

    # ---------------------------------------------------------- PPL + token-level KL(dense‖quant) + top-1 agreement
    @torch.no_grad()
    def compare(self, ids):
        nll, nt, top1 = 0.0, 0, 0
        kls = []
        for x in batched_windows(ids, self.kl_bs):
            x = x.to(self.device)
            ld = F.log_softmax(self.dense(x[:, :-1]).logits.float(), -1)
            lq = F.log_softmax(self.work(x[:, :-1]).logits.float(), -1)
            nll += -lq.gather(-1, x[:, 1:, None]).double().sum().item()
            nt += x.shape[0] * (x.shape[1] - 1)
            kls.append(F.kl_div(lq, ld, log_target=True, reduction="none").sum(-1).flatten().double().cpu())
            top1 += (ld.argmax(-1) == lq.argmax(-1)).sum().item()
            del ld, lq
        kl = torch.cat(kls).numpy()
        return {"ppl": float(math.exp(nll / nt)), "kl_mean": float(kl.mean()),
                "kl_p99": float(np.percentile(kl, 99)), "top1": float(100.0 * top1 / nt), "n_tokens": nt}

    # ---------------------------------------------------------- lm-eval-harness, 0-shot, per-question predictions kept
    def lmeval(self, limit=1000, bs=16, model=None):
        import lm_eval
        from lm_eval.models.huggingface import HFLM
        from lm_eval.tasks import TaskManager
        tasks = self.lmeval_tasks
        lm = HFLM(pretrained=model or self.work, tokenizer=self.tok, batch_size=bs)
        tm = TaskManager(include_path=self.lmeval_include) if self.lmeval_include else TaskManager()
        res = lm_eval.simple_evaluate(model=lm, tasks=list(tasks), num_fewshot=0, limit=limit, log_samples=True,
                                      bootstrap_iters=0, random_seed=0, numpy_random_seed=1234,
                                      torch_random_seed=1234, fewshot_random_seed=1234, task_manager=tm)
        out = {}
        for t in tasks:
            samples = sorted(res["samples"][t], key=lambda s: s["doc_id"])
            r = res["results"][t]
            d = {"acc": float(r["acc,none"]), "n": len(samples),
                 "doc_ids": [int(s["doc_id"]) for s in samples],
                 "correct": "".join(str(int(s["acc"])) for s in samples)}
            if "acc_norm,none" in r:
                d["acc_norm"] = float(r["acc_norm,none"])
                d["correct_norm"] = "".join(str(int(s["acc_norm"])) for s in samples)
            out[t] = d
        return out

    # ---------------------------------------------------------- golden set: greedy generations vs the dense ones
    @torch.no_grad()
    def generate(self, prompts, max_new_tokens=128, bs=25, model=None):
        model = model or self.work
        tok = self.tok
        side = tok.padding_side
        tok.padding_side = "left"
        order = sorted(range(len(prompts)), key=lambda i: len(tok(prompts[i]).input_ids))
        outs = [None] * len(prompts)
        stop = set()
        for e in (tok.eos_token_id, model.generation_config.eos_token_id):
            stop |= set(e if isinstance(e, (list, tuple)) else [e])
        stop.discard(None)
        try:
            for i in range(0, len(order), bs):
                idx = order[i:i + bs]
                enc = tok([prompts[j] for j in idx], return_tensors="pt", padding=True).to(self.device)
                gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, temperature=None,
                                     top_p=None, top_k=None, pad_token_id=tok.pad_token_id,
                                     eos_token_id=sorted(stop))
                for j, row in zip(idx, gen[:, enc.input_ids.shape[1]:].tolist()):
                    cut = next((k for k, t in enumerate(row) if t in stop), len(row))
                    outs[j] = row[:cut]
        finally:
            tok.padding_side = side
        return outs


def golden_prompts(tok, n=200):
    """200 instruction prompts from databricks-dolly-15k (no-context rows, fixed shuffle), Qwen3 chat template,
    thinking disabled."""
    from huggingface_hub import hf_hub_download
    p = hf_hub_download("databricks/databricks-dolly-15k", "databricks-dolly-15k.jsonl", repo_type="dataset")
    rows = [json.loads(l) for l in open(p)]
    perm = np.random.RandomState(0).permutation(len(rows))
    sel = [rows[i] for i in perm if not rows[i]["context"].strip()][:n]
    return [chat_prompt(tok, r["instruction"]) for r in sel]


def chat_prompt(tok, text):
    msgs = [{"role": "user", "content": text}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def golden_compare(tok, dense_outs, outs):
    from rouge_score import rouge_scorer
    sc = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)
    em, rl = [], []
    for a, b in zip(dense_outs, outs):
        em.append(int(a == b))
        rl.append(sc.score(tok.decode(a, skip_special_tokens=True), tok.decode(b, skip_special_tokens=True))["rougeL"].fmeasure)
    return {"exact": float(100 * np.mean(em)), "rougeL": float(np.mean(rl)), "per_prompt_exact": "".join(map(str, em)),
            "per_prompt_rougeL": [round(x, 4) for x in rl]}


def flips(dense_lm, lm):
    """Dutta et al. 2024: % of questions correct→incorrect and incorrect→correct vs dense (per task + pooled)."""
    out, tot = {}, [0, 0, 0]
    for t, d in dense_lm.items():
        q = lm[t]
        assert d["doc_ids"] == q["doc_ids"], t
        a, b = d["correct"], q["correct"]
        c2i = sum(1 for x, y in zip(a, b) if x == "1" and y == "0")
        i2c = sum(1 for x, y in zip(a, b) if x == "0" and y == "1")
        n = len(a)
        out[t] = {"c2i": 100.0 * c2i / n, "i2c": 100.0 * i2c / n}
        tot[0] += c2i; tot[1] += i2c; tot[2] += n
    out["all"] = {"c2i": 100.0 * tot[0] / tot[2], "i2c": 100.0 * tot[1] / tot[2]}
    return out


def full_eval(bench, P, dense_ref, prompts, cfg, log=print):
    """All §3 metrics for the weights currently loaded in bench.work."""
    t0 = time.time()
    m = {}
    m["wt2"] = bench.compare(P["wt_test"])
    m["pile"] = bench.compare(P["pile_test"])
    log(f"    ppl wt2 {m['wt2']['ppl']:.3f} pile {m['pile']['ppl']:.3f} | kl {m['wt2']['kl_mean']:.4f}/{m['pile']['kl_mean']:.4f} "
        f"({time.time() - t0:.0f}s)")
    if cfg.get("lmeval", True):
        lm = bench.lmeval(limit=cfg.get("lmeval_limit", 1000), bs=cfg.get("lmeval_bs", 16))
        m["lmeval"] = lm
        m["flips"] = flips(dense_ref["lmeval"], lm)
        log("    " + " ".join(f"{t}={lm[t]['acc']:.3f}" for t in lm) + f" flips c→i {m['flips']['all']['c2i']:.2f}% ({time.time() - t0:.0f}s)")
    if cfg.get("golden", True):
        outs = bench.generate(prompts, bs=cfg.get("gen_bs", 25))
        m["golden"] = golden_compare(bench.tok, dense_ref["golden_outs"], outs)
        log(f"    golden exact {m['golden']['exact']:.1f}% rougeL {m['golden']['rougeL']:.3f} ({time.time() - t0:.0f}s)")
    m["eval_seconds"] = time.time() - t0
    return m


def dense_eval(bench, P, prompts, cfg, log=print):
    t0 = time.time()
    bench.reset()
    d = {"wt2": {"ppl": bench.ppl(P["wt_test"], bench.dense)}, "pile": {"ppl": bench.ppl(P["pile_test"], bench.dense)}}
    log(f"  dense ppl wt2 {d['wt2']['ppl']:.3f} pile {d['pile']['ppl']:.3f} ({time.time() - t0:.0f}s)")
    if cfg.get("lmeval", True):
        d["lmeval"] = bench.lmeval(limit=cfg.get("lmeval_limit", 1000), bs=cfg.get("lmeval_bs", 16), model=bench.dense)
        log("  dense " + " ".join(f"{t}={v['acc']:.3f}" for t, v in d["lmeval"].items()))
    if cfg.get("golden", True):
        d["golden_outs"] = bench.generate(prompts, bs=cfg.get("gen_bs", 25), model=bench.dense)
    d["eval_seconds"] = time.time() - t0
    return d

In [ ]:
%%writefile qbench/results.py
import json
import os
import platform
import sys
import time


def now():
    return time.strftime("%H:%M:%S")


def make_logger(prefix=""):
    def log(*a):
        print(f"[{now()}]{prefix}", *a, flush=True)
    return log


class Results:
    """One JSON file per run, rewritten atomically after every finished model so a crash loses at most one model."""

    def __init__(self, path, resume_from=None):
        self.path = path
        self.data = {}
        for p in (resume_from, path):
            if p and os.path.exists(p):
                with open(p) as f:
                    self.data = json.load(f)
                break
        self.data.setdefault("models", {})

    def save(self):
        tmp = self.path + ".tmp"
        with open(tmp, "w") as f:
            json.dump(self.data, f)
        os.replace(tmp, self.path)

    def done(self, key):
        return key in self.data["models"] and "metrics" in self.data["models"][key]

    def put_model(self, key, entry):
        self.data["models"][key] = entry
        self.save()


def env_info():
    import torch
    info = {"python": sys.version.split()[0], "platform": platform.platform(), "torch": torch.__version__,
            "cuda": torch.version.cuda, "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}
    for mod in ("transformers", "lm_eval", "numpy", "datasets", "gguf"):
        try:
            info[mod] = __import__(mod).__version__
        except Exception:
            pass
    return info

In [ ]:
%%writefile qbench/llamacpp.py
"""llama.cpp + imatrix rival: HF -> GGUF bf16 (round trip checked) -> imatrix on the shared calibration text ->
llama-quantize per format -> dequantize to F32 with llama-quantize -> load the 196 linears into the bench."""
import os
import subprocess
import sys

import torch

FORMATS = ["IQ2_XXS", "IQ2_XS", "IQ2_M", "Q2_K", "IQ3_XXS", "IQ3_M", "Q3_K_M", "IQ4_XS", "Q4_K_M"]
HANDOVER_REF = {"IQ2_XXS": (2.11, 111.7, 452.6), "IQ2_XS": (2.34, 54.3, 208.3), "IQ2_M": (2.76, 20.7, 82.4),
                "Q2_K": (2.95, 16.05, 53.0), "IQ3_XXS": (3.01, 15.37, 49.5), "IQ3_M": (3.67, 11.33, 33.4),
                "Q3_K_M": (3.87, 11.06, 31.4), "IQ4_XS": (4.25, 10.04, 29.25), "Q4_K_M": (4.78, 10.00, 29.0)}
GGUF_NAME = {"self_attn.q_proj": "attn_q", "self_attn.k_proj": "attn_k", "self_attn.v_proj": "attn_v",
             "self_attn.o_proj": "attn_output", "mlp.gate_proj": "ffn_gate", "mlp.up_proj": "ffn_up",
             "mlp.down_proj": "ffn_down"}


def gguf_name(hf_name):
    _, _, i, rest = hf_name.split(".", 3)
    return f"blk.{i}.{GGUF_NAME[rest]}.weight"


class LlamaCpp:
    def __init__(self, root, workdir, log=print):
        self.root, self.work, self.log = root, workdir, log
        os.makedirs(workdir, exist_ok=True)
        sys.path.insert(0, os.path.join(root, "gguf-py"))
        self.threads = str(os.cpu_count() or 4)

    def bin(self, name):
        return os.path.join(self.root, "build", "bin", name)

    def sh(self, *cmd):
        self.log("  $ " + " ".join(cmd))
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout[-3000:], r.stderr[-3000:], flush=True)
            raise RuntimeError(f"command failed: {cmd[0]}")
        return r

    def path(self, name):
        return os.path.join(self.work, name)

    def tensors(self, gguf_path, names):
        from gguf import GGUFReader
        r = GGUFReader(gguf_path)
        by = {t.name: t for t in r.tensors}
        return {n: by[gguf_name(n)] for n in names}

    def convert(self, snapshot, dense, names):
        bf16 = self.path("model-bf16.gguf")
        if not os.path.exists(bf16):
            self.sh(sys.executable, os.path.join(self.root, "convert_hf_to_gguf.py"), snapshot, "--outtype", "bf16",
                    "--outfile", bf16)
        # round trip: bf16 GGUF -> F32 must equal the HF weights bit for bit (no q/k row permutation for Qwen)
        f32 = self.path("model-bf16-as-f32.gguf")
        self.sh(self.bin("llama-quantize"), "--allow-requantize", bf16, f32, "F32", self.threads)
        mods = dict(dense.named_modules())
        bad = []
        for n, t in self.tensors(f32, names).items():
            W = torch.from_numpy(t.data.copy()).reshape(mods[n].weight.shape)
            if not torch.equal(W, mods[n].weight.detach().float().cpu()):
                bad.append(n)
        os.remove(f32)
        if bad:
            raise RuntimeError(f"GGUF round trip is NOT exact for {len(bad)} linears, e.g. {bad[:3]}")
        self.log("  GGUF round trip exact for all linears")
        return bf16

    def imatrix(self, bf16, calib_text):
        out = self.path("imatrix.gguf")
        if not os.path.exists(out):
            txt = self.path("calib.txt")
            with open(txt, "w") as f:
                f.write(calib_text)
            self.sh(self.bin("llama-imatrix"), "-m", bf16, "-f", txt, "-o", out, "-c", "512", "--chunks", "32",
                    "-t", self.threads)
        return out

    def quantize(self, bf16, imat, fmt, names, shapes):
        q = self.path(f"q-{fmt}.gguf")
        self.sh(self.bin("llama-quantize"), "--imatrix", imat, bf16, q, fmt, self.threads)
        tq = self.tensors(q, names)
        nbytes = sum(int(t.n_bytes) for t in tq.values())
        types = {}
        for t in tq.values():
            types[t.tensor_type.name] = types.get(t.tensor_type.name, 0) + 1
        f32 = self.path(f"q-{fmt}-f32.gguf")
        self.sh(self.bin("llama-quantize"), "--allow-requantize", q, f32, "F32", self.threads)
        W = {n: torch.from_numpy(t.data.copy()).reshape(shapes[n]) for n, t in self.tensors(f32, names).items()}
        os.remove(q)
        os.remove(f32)
        return W, nbytes, types

In [ ]:
%%writefile qbench/gptq_worker.py
"""Runs the official GPTQModel pipeline in a subprocess and writes the effective (dequantized) weights of the
196 linears. Dequantization is format-agnostic: each quantized module is probed with the identity matrix
(y = I·Wᵀ), i.e. exactly what its own torch kernel computes at inference."""
import argparse
import json
import os
import shutil

import torch


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--calib", required=True)
    ap.add_argument("--names", required=True)
    ap.add_argument("--bits", type=int, required=True)
    ap.add_argument("--sym", type=int, required=True)
    ap.add_argument("--desc_act", type=int, required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--tmp", required=True)
    a = ap.parse_args()
    from gptqmodel import BACKEND, GPTQModel, QuantizeConfig
    from safetensors import safe_open

    names = json.load(open(a.names))
    calib = torch.load(a.calib)
    data = [{"input_ids": r.tolist(), "attention_mask": [1] * len(r)} for r in calib]
    qc = QuantizeConfig(bits=a.bits, group_size=128, sym=bool(a.sym), desc_act=bool(a.desc_act))
    model = GPTQModel.load(a.model, qc)
    model.quantize(data, batch_size=1)
    shutil.rmtree(a.tmp, ignore_errors=True)
    model.save(a.tmp)
    del model
    torch.cuda.empty_cache()

    nbytes = 0
    for f in os.listdir(a.tmp):
        if f.endswith(".safetensors"):
            with safe_open(os.path.join(a.tmp, f), "pt") as st:
                for k in st.keys():
                    if any(k.startswith(n + ".") for n in names):
                        t = st.get_slice(k)
                        n_el = 1
                        for s in t.get_shape():
                            n_el *= s
                        nbytes += n_el * torch.empty((), dtype=getattr(torch, {"I32": "int32", "F16": "float16",
                                  "BF16": "bfloat16", "F32": "float32", "I16": "int16", "I8": "int8",
                                  "U8": "uint8", "I64": "int64"}[t.get_dtype()])).element_size()

    qm = GPTQModel.load(a.tmp, backend=BACKEND.GPTQ_TORCH, device="cuda:0")
    hf = qm.model
    W = {}
    with torch.no_grad():
        for n in names:
            mod = hf.get_submodule(n)
            n_in = mod.in_features
            dt = getattr(mod, "scales", torch.empty(0, dtype=torch.float16)).dtype
            y = mod(torch.eye(n_in, dtype=dt, device="cuda:0"))
            W[n] = y.float().T.contiguous().cpu()
    torch.save({"weights": W, "bytes": nbytes}, a.out)
    shutil.rmtree(a.tmp, ignore_errors=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile qbench/awq_worker.py
"""Official AutoAWQ 0.2.9 pipeline (run with PYTHONPATH pointing to an isolated site with transformers 4.51.3).
Scales/clipping are searched and applied by AutoAWQ (export_compatible=True), then every linear is quantized with
AutoAWQ's own pseudo_quantize_tensor (the exact values its GEMM kernel would dequantize). The norm rescaling AWQ
introduces is folded back into the linears so that norms stay identical to the dense model (same function)."""
import argparse
import json

import torch


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True)
    ap.add_argument("--calib", required=True)
    ap.add_argument("--names", required=True)
    ap.add_argument("--bits", type=int, required=True)
    ap.add_argument("--out", required=True)
    a = ap.parse_args()
    from awq import AutoAWQForCausalLM
    from transformers import AutoTokenizer

    names = json.load(open(a.names))
    tok = AutoTokenizer.from_pretrained(a.model)
    model = AutoAWQForCausalLM.from_pretrained(a.model)
    hf = model.model
    lin = set(names)
    before = {n: p.detach().clone().float().cpu() for n, p in hf.named_parameters()
              if n.rsplit(".", 1)[0] not in lin}
    calib = [r.tolist() for r in torch.load(a.calib)]
    model.quantize(tok, quant_config={"zero_point": True, "q_group_size": 128, "w_bit": a.bits, "version": "GEMM"},
                   calib_data=calib, max_calib_samples=len(calib), max_calib_seq_len=512, export_compatible=True)
    q = model.quantizer
    W = {}
    with torch.no_grad():
        for n in names:
            w = hf.get_submodule(n).weight.data.to("cuda:0").half()
            W[n] = q.pseudo_quantize_tensor(w)[0].float().cpu()
        after = {n: p.detach().float().cpu() for n, p in hf.named_parameters() if n.rsplit(".", 1)[0] not in lin}
        changed = [n for n in before if not torch.equal(before[n], after[n])]
        for n in changed:
            if not n.endswith(("input_layernorm.weight", "post_attention_layernorm.weight")):
                raise RuntimeError(f"AWQ modified an unexpected non-linear parameter: {n}")
            r = after[n] / before[n]
            layer = n.rsplit(".", 2)[0]
            outs = ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj"] if "input_layernorm" in n \
                else ["mlp.gate_proj", "mlp.up_proj"]
            for o in outs:
                W[f"{layer}.{o}"] = W[f"{layer}.{o}"] * r[None, :]
    g = 128
    bpw = a.bits + (16 + a.bits) / g  # GEMM layout: packed codes + fp16 scale + packed zero per group of 128
    torch.save({"weights": W, "bpw": bpw, "folded_norms": changed}, a.out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile run_phase1.py
#!/usr/bin/env python
"""Phase 1: control, V1 (log scales, g128 and g64), V2 (ternary), V3 (V1 + ternary) for one rotation seed.

  --screen_only   picks (k, bz) for the V1 log scales on calibration data (seed-0 rotations) and exits
  --preflight     fast end-to-end check of every code path (minutes) before committing GPU hours
"""
import argparse
import gc
import json
import math
import os
import sys
import time

import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from qbench import bench as B
from qbench import hessian, mixed
from qbench import protocol as PR
from qbench import quant as Q
from qbench.hadamard import rotation_signs
from qbench.results import Results, env_info, make_logger

POINTS = [2.04, 2.3, 2.75, 3.0, 3.5, 3.9, 4.25]
SCREEN = [(6, 4), (6, 6), (8, 4), (8, 6)]


def get_args(argv=None):
    return build_parser().parse_args(argv)


def build_parser():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default=PR.MP)
    ap.add_argument("--device", default="cuda:0")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--out", default="out")
    ap.add_argument("--cache", default="cache")
    ap.add_argument("--resume_from", default=None)
    ap.add_argument("--points", default=",".join(map(str, POINTS)))
    ap.add_argument("--variants", default="control,V2,V1,V3,V1g64")
    ap.add_argument("--v1_kbz", default="auto", help="'auto' (read screening file) or e.g. '8,6'")
    ap.add_argument("--screen_only", action="store_true")
    ap.add_argument("--preflight", action="store_true")
    ap.add_argument("--no_lmeval", action="store_true")
    ap.add_argument("--no_golden", action="store_true")
    ap.add_argument("--lmeval_limit", type=int, default=1000)
    ap.add_argument("--lmeval_bs", type=int, default=16)
    ap.add_argument("--lmeval_tasks", default=",".join(B.LMEVAL_TASKS))
    ap.add_argument("--lmeval_include", default=None)
    ap.add_argument("--fake_data", action="store_true", help="synthetic text instead of Pile/wikitext/dolly (tests)")
    ap.add_argument("--max_layers", type=int, default=0, help="tests only: quantize the first N layers")
    return ap


# ---------------------------------------------------------------- setup shared by every phase
def setup(args, log):
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    os.makedirs(args.out, exist_ok=True)
    os.makedirs(args.cache, exist_ok=True)
    tok, dense = PR.load_model(args.model, args.device)
    names = PR.linear_names(dense.config.num_hidden_layers)
    if args.max_layers:
        names = names[:7 * args.max_layers]
    raw = None
    if args.fake_data:
        from tests.fake import fake_corpus
        raw = fake_corpus()
    cache = os.path.join(args.cache, "protocol_fake.pt" if args.fake_data else "protocol.pt")
    P = PR.load_protocol(tok, cache_path=cache, raw=raw)
    bench = B.Bench(dense, tok, names, args.device, lmeval_tasks=args.lmeval_tasks.split(","),
                    lmeval_include=args.lmeval_include)
    if args.fake_data:
        from tests.fake import fake_prompts
        prompts = fake_prompts(tok)
    elif args.no_golden:
        prompts = []
    else:
        prompts = B.golden_prompts(tok)
    cfg = {"lmeval": not args.no_lmeval, "golden": not args.no_golden, "lmeval_limit": args.lmeval_limit,
           "lmeval_bs": args.lmeval_bs}
    log(f"setup done: {len(names)} linears, device {args.device}, protocol {PR.fingerprint(P)}")
    return tok, dense, names, P, bench, prompts, cfg


def check_dense(d, fake, log):
    dev = {k: abs(d[k]["ppl"] - PR.DENSE_REF[k]) / PR.DENSE_REF[k] for k in PR.DENSE_REF}
    log(f"dense vs HANDOVER reference (wt2 27.15 / Pile 9.38): rel. dev {dev}")
    if fake:
        return dev
    if max(dev.values()) > 0.03:
        raise SystemExit(f"PROTOCOL MISMATCH: dense ppl {d['wt2']['ppl']:.3f}/{d['pile']['ppl']:.3f} "
                         f"vs 27.15/9.38. Stop and check versions/data before spending GPU time.")
    if max(dev.values()) > 0.005:
        log("WARNING: dense ppl deviates >0.5% from the HANDOVER reference")
    return dev


def get_dense(res, bench, P, prompts, cfg, args, log):
    if "dense" not in res.data:
        res.data["dense"] = B.dense_eval(bench, P, prompts, cfg, log)
        res.data["dense_check"] = check_dense(res.data["dense"], args.fake_data, log)
        res.save()
    return res.data["dense"]


def get_hessians(dense, P, names, args, log):
    t0 = time.time()
    H = hessian.collect_hessians(dense, P["calib"], names, args.device)
    log(f"hessians: {len(names)} ({time.time() - t0:.0f}s)")
    return H


def quantize_all(dense, H, names, opts, seed, device, log):
    mods = dict(dense.named_modules())
    store, t0 = {m: {} for m in names}, time.time()
    for j, m in enumerate(names):
        W = mods[m].weight.data.to(device)
        signs = rotation_signs(W.shape[1], seed, j)
        Wr, Hr = Q.rotate(W, H[m], signs)
        Hinv, dead = Q.prep_hinv(Hr)
        for opt in opts:
            store[m][opt.name] = Q.quantize_matrix(Wr, Hr, Hinv, dead, opt, signs)
        del Wr, Hr, Hinv
        if (j + 1) % 28 == 0 or j + 1 == len(names):
            log(f"  quantized {j + 1}/{len(names)} matrices x {len(opts)} options ({time.time() - t0:.0f}s)")
    return store


def bits_table(dense, names, opts):
    mods = dict(dense.named_modules())
    bits, N = {}, 0
    for m in names:
        r, n = mods[m].weight.shape
        N += r * n
        bits[m] = {o.name: Q.opt_bits(o, r, n) for o in opts}
    return bits, N


def weights_for(store, choice, device):
    return {m: store[m][o].weff(device) for m, o in choice.items()}


# ---------------------------------------------------------------- screening of (k, bz) for V1
def screen(args, log):
    tok, dense, names, P, bench, prompts, cfg = setup(args, log)
    H = get_hessians(dense, P, names, args, log)
    opts = [Q.Opt("q", b) for b in (2, 3)] + [Q.Opt("q", b, "log", k, bz, 128) for (k, bz) in SCREEN for b in (2, 3)]
    store = quantize_all(dense, H, names, opts, 0, args.device, log)
    bits, N = bits_table(dense, names, opts)
    L, bpw = {}, {}
    for o in opts:
        bench.load(weights_for(store, {m: o.name for m in names}, args.device))
        L[o.name] = bench.logppl(P["sens_ids"])
        bpw[o.name] = sum(bits[m][o.name] for m in names) / N
        log(f"  screen {o.name:22s} bpw {bpw[o.name]:.4f} calib log-ppl {L[o.name]:.5f}")
    lam = (L["q2"] - L["q3"]) / (bpw["q3"] - bpw["q2"])
    score = {}
    for key in [None] + SCREEN:
        tot = 0.0
        for b in (2, 3):
            name = f"q{b}" if key is None else Q.Opt("q", b, "log", key[0], key[1], 128).name
            tot += L[name] + lam * (bpw[name] - bpw[f"q{b}"])
        score["fp16" if key is None else f"k{key[0]}z{key[1]}"] = tot
    best = min(SCREEN, key=lambda kb: score[f"k{kb[0]}z{kb[1]}"])
    out = {"choice": list(best), "lambda_logppl_per_bpw": lam, "score_at_equal_bpw": score,
           "calib_logppl": L, "bpw": bpw, "env": env_info(), "protocol": PR.fingerprint(P)}
    with open(os.path.join(args.out, "v1_screening.json"), "w") as f:
        json.dump(out, f, indent=1)
    log(f"screening: chose k={best[0]} bz={best[1]}; scores {score}")


# ---------------------------------------------------------------- main run for one seed
def run(args, log):
    if args.v1_kbz == "auto":
        with open(os.path.join(args.out, "v1_screening.json")) as f:
            k, bz = json.load(f)["choice"]
    else:
        k, bz = map(int, args.v1_kbz.split(","))
    points = [float(p) for p in args.points.split(",")]
    variants = args.variants.split(",")
    tok, dense, names, P, bench, prompts, cfg = setup(args, log)
    res = Results(os.path.join(args.out, f"phase1_seed{args.seed}.json"), args.resume_from)
    res.data["meta"] = {"seed": args.seed, "device": args.device, "v1_kbz": [k, bz], "points": points,
                        "variants": variants, "env": env_info(), "protocol": PR.fingerprint(P), "cfg": cfg,
                        "n_linears": len(names)}
    res.save()
    dense_ref = get_dense(res, bench, P, prompts, cfg, args, log)
    H = get_hessians(dense, P, names, args, log)

    F = [Q.Opt("q", 2), Q.Opt("q", 3), Q.Opt("q", 4), Q.Opt("tern")]
    L = [Q.Opt("q", b, "log", k, bz, 128) for b in (2, 3, 4)]
    L64 = [Q.Opt("q", b, "log", k, bz, 64) for b in (2, 3, 4)]
    TL = [Q.Opt("tern", 0, "log", k, bz, 128)]
    opts = list(F)
    if {"V1", "V3"} & set(variants):
        opts += L
    if "V1g64" in variants:
        opts += L64
    if "V3" in variants:
        opts += TL
    log(f"quantizing seed {args.seed}: options {[o.name for o in opts]}")
    store = quantize_all(dense, H, names, opts, args.seed, args.device, log)
    del H
    gc.collect()
    bits, N = bits_table(dense, names, opts)
    res.data["n_weights"] = N

    fam = {"V2": ([o.name for o in F], "q4"), "V1": ([o.name for o in L], L[2].name),
           "V1g64": ([o.name for o in L64], L64[2].name), "V3": ([o.name for o in TL + L], L[2].name)}
    sens = res.data.setdefault("sensitivity", {})
    for v in ("V2", "V1", "V1g64", "V3"):
        if v not in variants or v in sens:
            continue
        o_names, ref = fam[v]
        t0 = time.time()
        if v == "V3" and "V1" in sens:
            base, D = mixed.sensitivity(bench, store, names, [TL[0].name, ref], ref, P["sens_ids"], log)
            for m in names:
                D[m].update(sens["V1"]["delta"][m])
        else:
            base, D = mixed.sensitivity(bench, store, names, o_names, ref, P["sens_ids"], log)
        sens[v] = {"ref": ref, "base_logppl": base, "delta": D, "seconds": time.time() - t0}
        res.save()
        log(f"sensitivity {v}: {time.time() - t0:.0f}s")

    jobs = []
    if "control" in variants:
        for o in ("q2", "q3", "q4"):
            jobs.append(("control", o, {m: o for m in names}))
    for v in ("V2", "V1", "V3", "V1g64"):
        if v not in variants:
            continue
        o_names, _ = fam[v]
        for p in points:
            choice, tot = mixed.knapsack(names, o_names, bits, sens[v]["delta"], p * N)
            key = f"{v}@{p:.2f}#s{args.seed}"
            if choice is None:
                res.put_model(key, {"method": v, "target_bpw": p, "seed": args.seed, "infeasible": True,
                                    "min_bpw": tot / N, "metrics": None})
                log(f"{key}: infeasible (min {tot / N:.4f} bpw)")
                continue
            jobs.append((v, p, choice))

    t_start = time.time()
    for i, (v, p, choice) in enumerate(jobs):
        key = f"{v}@{p}#s{args.seed}" if v == "control" else f"{v}@{p:.2f}#s{args.seed}"
        if res.done(key):
            continue
        tot = sum(bits[m][choice[m]] for m in names)
        alloc = {}
        for o in choice.values():
            alloc[o] = alloc.get(o, 0) + 1
        log(f"[{i + 1}/{len(jobs)}] {key}: bpw {tot / N:.4f} alloc {alloc}")
        bench.load(weights_for(store, choice, args.device))
        metrics = B.full_eval(bench, P, dense_ref, prompts, cfg, log)
        res.put_model(key, {"method": v, "target_bpw": p, "seed": args.seed, "bpw": tot / N, "alloc": alloc,
                            "choice": choice, "metrics": metrics})
        el = time.time() - t_start
        log(f"    done; elapsed {el / 60:.1f} min, ~{el / (i + 1) * (len(jobs) - i - 1) / 60:.0f} min left")
    log("PHASE 1 DONE")


def preflight(args, log):
    """Every code path on a tiny budget: dense check, lm-eval/golden on few items, 2 layers quantized, all options,
    sensitivity + knapsack + one full eval."""
    t0 = time.time()
    tok, dense, names, P, bench, prompts, cfg = setup(args, log)
    cfg = dict(cfg, lmeval_limit=min(5, args.lmeval_limit))
    prompts = prompts[:4]
    d = B.dense_eval(bench, P, prompts, cfg, log)
    check_dense(d, args.fake_data, log)
    sub = names[:14]
    Hs = hessian.collect_hessians(dense, P["calib"][:2], sub, args.device)
    opts = [Q.Opt("q", 2), Q.Opt("q", 4), Q.Opt("tern"), Q.Opt("q", 2, "log", 8, 6, 128),
            Q.Opt("q", 4, "log", 8, 6, 128), Q.Opt("q", 2, "log", 8, 6, 64), Q.Opt("tern", 0, "log", 8, 6, 128)]
    store = quantize_all(dense, Hs, sub, opts, 0, args.device, log)
    bits, N = bits_table(dense, sub, opts)
    bench.reset()
    bench_sub = B.Bench(dense, tok, sub, args.device, lmeval_tasks=bench.lmeval_tasks, lmeval_include=bench.lmeval_include)
    del bench
    base, D = mixed.sensitivity(bench_sub, store, sub, ["q2", "q4", "tern"], "q4", P["sens_ids"], log)
    choice, tot = mixed.knapsack(sub, ["tern", "q2", "q4"], bits, D, 3.0 * N)
    log(f"knapsack ok: {tot / N:.3f} bpw")
    bench_sub.load(weights_for(store, choice, args.device))
    m = B.full_eval(bench_sub, P, d, prompts, cfg, log)
    assert m["wt2"]["ppl"] > 0 and m["wt2"]["kl_mean"] > 0
    log(f"PREFLIGHT OK ({time.time() - t0:.0f}s)")


if __name__ == "__main__":
    a = get_args()
    lg = make_logger(f"[s{a.seed}]")
    if a.preflight:
        preflight(a, lg)
    elif a.screen_only:
        screen(a, lg)
    else:
        run(a, lg)

In [ ]:
%%writefile run_phase2.py
#!/usr/bin/env python
"""Phase 2: rivals with their official pipelines and the shared calibration, evaluated on the same bench.
  --part llamacpp   llama.cpp imatrix formats (needs a built llama.cpp at --llamacpp)
  --part gptq       GPTQModel 4/3/2 bits g128; desc_act and sym chosen on calibration data (not on eval data)
  --part awq        AutoAWQ 4 and 3 bits g128 (isolated site at --awq_site)
"""
import json
import os
import subprocess
import sys
import time

import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import run_phase1 as P1
from qbench import bench as B
from qbench import protocol as PR
from qbench.llamacpp import FORMATS, HANDOVER_REF, LlamaCpp
from qbench.results import Results, env_info, make_logger

HERE = os.path.dirname(os.path.abspath(__file__))


def extra_args(ap):
    ap.add_argument("--part", required=True, choices=["llamacpp", "gptq", "awq"])
    ap.add_argument("--llamacpp", default="/tmp/llama.cpp")
    ap.add_argument("--awq_site", default="/tmp/site_awq")
    ap.add_argument("--formats", default=",".join(FORMATS))
    ap.add_argument("--gptq_bits", default="4,3,2")
    ap.add_argument("--awq_bits", default="4,3")
    ap.add_argument("--work", default="/tmp/qwork")


def evaluate(res, key, entry, W, bench, P, dense_ref, prompts, cfg, log):
    bench.load(W)
    log(f"{key}: bpw {entry['bpw']:.4f}")
    entry["metrics"] = B.full_eval(bench, P, dense_ref, prompts, cfg, log)
    res.put_model(key, entry)


def worker(script, args, gpu, extra_env=None):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    env.update(extra_env or {})
    cmd = [sys.executable, os.path.join(HERE, "qbench", script)] + args
    r = subprocess.run(cmd, env=env)
    if r.returncode != 0:
        raise RuntimeError(f"{script} failed ({r.returncode})")


def main():
    ap = P1.build_parser()
    extra_args(ap)
    args = ap.parse_args()
    log = make_logger(f"[{args.part}]")
    gpu = int(args.device.split(":")[1]) if ":" in args.device else 0
    tok, dense, names, P, bench, prompts, cfg = P1.setup(args, log)
    res = Results(os.path.join(args.out, f"phase2_{args.part}.json"), args.resume_from)
    res.data["meta"] = {"part": args.part, "env": env_info(), "protocol": PR.fingerprint(P), "cfg": cfg}
    res.save()
    dense_ref = P1.get_dense(res, bench, P, prompts, cfg, args, log)
    mods = dict(dense.named_modules())
    shapes = {n: tuple(mods[n].weight.shape) for n in names}
    N = sum(r * c for r, c in shapes.values())
    snap = PR.model_snapshot(args.model)
    os.makedirs(args.work, exist_ok=True)
    calib_pt = os.path.join(args.work, "calib.pt")
    torch.save(P["calib"].clone(), calib_pt)
    names_json = os.path.join(args.work, "names.json")
    json.dump(names, open(names_json, "w"))

    if args.part == "llamacpp":
        lc = LlamaCpp(args.llamacpp, os.path.join(args.work, "gguf"), log)
        bf16 = lc.convert(snap, dense, names)
        imat = lc.imatrix(bf16, P["calib_text"])
        res.data["llamacpp_commit"] = subprocess.run(["git", "-C", args.llamacpp, "describe", "--tags"],
                                                     capture_output=True, text=True).stdout.strip()
        for fmt in args.formats.split(","):
            key = f"llama.cpp {fmt}"
            if res.done(key):
                continue
            W, nbytes, types = lc.quantize(bf16, imat, fmt, names, shapes)
            entry = {"method": "llama.cpp", "format": fmt, "bpw": nbytes * 8 / N, "tensor_types": types,
                     "handover_ref": HANDOVER_REF.get(fmt)}
            evaluate(res, key, entry, W, bench, P, dense_ref, prompts, cfg, log)
            ref = HANDOVER_REF.get(fmt)
            if ref:
                m = res.data["models"][key]["metrics"]
                log(f"    control vs HANDOVER: bpw {entry['bpw']:.2f}/{ref[0]} pile {m['pile']['ppl']:.2f}/{ref[1]} "
                    f"wt2 {m['wt2']['ppl']:.2f}/{ref[2]}")

    elif args.part == "gptq":
        search = res.data.setdefault("gptq_search", {})
        for b in map(int, args.gptq_bits.split(",")):
            key = f"GPTQ {b}bit g128"
            if res.done(key):
                continue
            best = None
            for desc in (0, 1):
                for sym in (1, 0):
                    tag = f"{b}bit desc_act={desc} sym={sym}"
                    out = os.path.join(args.work, "gptq_w.pt")
                    t0 = time.time()
                    worker("gptq_worker.py", ["--model", snap, "--calib", calib_pt, "--names", names_json,
                                              "--bits", str(b), "--sym", str(sym), "--desc_act", str(desc),
                                              "--out", out, "--tmp", os.path.join(args.work, "gptq_tmp")], gpu)
                    d = torch.load(out)
                    bench.load(d["weights"])
                    lp = bench.logppl(P["sens_ids"])
                    search[tag] = {"calib_logppl": lp, "bpw": d["bytes"] * 8 / N, "seconds": time.time() - t0}
                    res.save()
                    log(f"  {tag}: calib log-ppl {lp:.5f} bpw {d['bytes'] * 8 / N:.4f}")
                    if best is None or lp < best[0]:
                        best = (lp, tag, d)
                    else:
                        del d
            lp, tag, d = best
            entry = {"method": "GPTQModel", "bits": b, "config": tag, "bpw": d["bytes"] * 8 / N,
                     "selected_on": "calibration log-ppl (sens_ids)"}
            evaluate(res, key, entry, d["weights"], bench, P, dense_ref, prompts, cfg, log)
            del best, d

    elif args.part == "awq":
        for b in map(int, args.awq_bits.split(",")):
            key = f"AWQ {b}bit g128"
            if res.done(key):
                continue
            out = os.path.join(args.work, "awq_w.pt")
            worker("awq_worker.py", ["--model", snap, "--calib", calib_pt, "--names", names_json, "--bits", str(b),
                                     "--out", out], gpu, {"PYTHONPATH": args.awq_site})
            d = torch.load(out)
            entry = {"method": "AutoAWQ", "bits": b, "bpw": d["bpw"], "folded_norms": len(d["folded_norms"])}
            evaluate(res, key, entry, d["weights"], bench, P, dense_ref, prompts, cfg, log)
    log(f"PHASE 2 {args.part} DONE")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile aggregate.py
#!/usr/bin/env python
"""Builds the HANDOVER §7 tables from phase1_seed*.json and phase2_*.json (any folder layout below --inp).
Output: tables.md, tables.csv, summary.json. Seeds are aggregated as mean ± std."""
import argparse
import glob
import json
import os
from collections import defaultdict

import numpy as np

TASKS = ["arc_easy", "arc_challenge", "hellaswag", "piqa", "winogrande"]
COLS = ["bpw", "ppl_wt2", "ppl_pile", "kl_wt2", "kl_pile", "klp99_wt2", "klp99_pile", "top1_wt2", "top1_pile"] + \
       [f"acc_{t}" for t in TASKS] + ["acc_mean", "flip_c2i", "flip_i2c", "gold_exact", "gold_rougeL"]


def flat(entry):
    m = entry["metrics"]
    r = {"bpw": entry.get("bpw")}
    for s in ("wt2", "pile"):
        r[f"ppl_{s}"] = m[s]["ppl"]
        r[f"kl_{s}"] = m[s].get("kl_mean", 0.0)
        r[f"klp99_{s}"] = m[s].get("kl_p99", 0.0)
        r[f"top1_{s}"] = m[s].get("top1", 100.0)
    if "lmeval" in m:
        accs = [m["lmeval"][t]["acc"] for t in TASKS if t in m["lmeval"]]
        for t in TASKS:
            if t in m["lmeval"]:
                r[f"acc_{t}"] = m["lmeval"][t]["acc"]
        r["acc_mean"] = float(np.mean(accs))
    if "flips" in m:
        r["flip_c2i"], r["flip_i2c"] = m["flips"]["all"]["c2i"], m["flips"]["all"]["i2c"]
    if "golden" in m:
        r["gold_exact"], r["gold_rougeL"] = m["golden"]["exact"], m["golden"]["rougeL"]
    return r


def dense_row(d):
    r = {"bpw": 16.0, "kl_wt2": 0.0, "kl_pile": 0.0, "klp99_wt2": 0.0, "klp99_pile": 0.0, "top1_wt2": 100.0,
         "top1_pile": 100.0, "ppl_wt2": d["wt2"]["ppl"], "ppl_pile": d["pile"]["ppl"], "flip_c2i": 0.0,
         "flip_i2c": 0.0, "gold_exact": 100.0, "gold_rougeL": 1.0}
    if "lmeval" in d:
        for t in TASKS:
            if t in d["lmeval"]:
                r[f"acc_{t}"] = d["lmeval"][t]["acc"]
        r["acc_mean"] = float(np.mean([d["lmeval"][t]["acc"] for t in TASKS if t in d["lmeval"]]))
    return r


def fmt(vals, col):
    vals = [v for v in vals if v is not None]
    if not vals:
        return ""
    nd = 2 if col.startswith(("ppl", "top1", "flip", "gold_exact")) else (4 if col.startswith(("kl", "bpw")) else 3)
    if col.startswith("acc"):
        vals = [100 * v for v in vals]
        nd = 1
    mu = np.mean(vals)
    return f"{mu:.{nd}f}" + (f" ± {np.std(vals, ddof=1):.{nd}f}" if len(vals) > 1 else "")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--inp", nargs="+", default=["."])
    ap.add_argument("--out", default="tables")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    files = sorted({f for d in a.inp for f in glob.glob(os.path.join(d, "**", "phase*.json"), recursive=True)})
    groups, order, dense, infeasible = defaultdict(list), [], None, []
    for f in files:
        D = json.load(open(f))
        if dense is None and "dense" in D:
            dense = D["dense"]
        for key, e in D.get("models", {}).items():
            if e.get("infeasible"):
                infeasible.append(f"{e['method']} @ {e['target_bpw']} bpw (min {e['min_bpw']:.3f})")
                continue
            if not e.get("metrics"):
                continue
            g = key.split("#s")[0]
            if g not in groups:
                order.append(g)
            groups[g].append(flat(e))
    rows = [("dense (fp32)", [dense_row(dense)])] if dense else []
    rows += [(g, groups[g]) for g in sorted(order, key=lambda g: (np.mean([r["bpw"] for r in groups[g]]), g))]

    def table(cols, title):
        out = [f"### {title}", "", "| method | n | " + " | ".join(cols) + " |", "|---|---|" + "---|" * len(cols)]
        for name, rs in rows:
            out.append(f"| {name} | {len(rs)} | " + " | ".join(fmt([r.get(c) for r in rs], c) for c in cols) + " |")
        return "\n".join(out) + "\n"

    md = ["# Qwen3-0.6B — quantization benchmark (HANDOVER protocol)", "",
          "Mean ± std over rotation seeds (n = number of seeds/runs). acc in %, KL in nats/token, top-1 in %, "
          "flips in % of questions vs dense. Rows sorted by effective bpw (196 block linears only).", "",
          table(["bpw", "ppl_wt2", "ppl_pile"], "Perplexity"),
          table(["bpw", "kl_wt2", "klp99_wt2", "top1_wt2", "kl_pile", "klp99_pile", "top1_pile"], "KL vs dense"),
          table(["bpw"] + [f"acc_{t}" for t in TASKS] + ["acc_mean"], "lm-eval 0-shot accuracy"),
          table(["bpw", "flip_c2i", "flip_i2c", "acc_mean"], "Flips (Dutta et al. 2024)"),
          table(["bpw", "gold_exact", "gold_rougeL"], "Golden set (proxy: greedy 128 tok vs dense)")]
    if infeasible:
        md += ["### Infeasible budgets", ""] + [f"- {x}" for x in infeasible]
    open(os.path.join(a.out, "tables.md"), "w").write("\n".join(md))
    with open(os.path.join(a.out, "tables.csv"), "w") as f:
        f.write("method,n," + ",".join(COLS) + "\n")
        for name, rs in rows:
            f.write(f"\"{name}\",{len(rs)}," + ",".join(
                str(np.mean([r[c] for r in rs if r.get(c) is not None])) if any(r.get(c) is not None for r in rs)
                else "" for c in COLS) + "\n")
    json.dump({"files": files, "rows": {n: rs for n, rs in rows}, "infeasible": infeasible},
              open(os.path.join(a.out, "summary.json"), "w"), indent=1)
    print("\n".join(md))


if __name__ == "__main__":
    main()

In [ ]:
import subprocess, time, sys, torch
def launch(jobs):
    """jobs: list of (name, shell command). Runs them in parallel, prints log tails, fails loudly."""
    procs = {n: subprocess.Popen(c, shell=True, stdout=open(f"/kaggle/working/logs/{n}.log", "w"),
                                 stderr=subprocess.STDOUT) for n, c in jobs}
    last = 0
    while any(p.poll() is None for p in procs.values()):
        time.sleep(30)
        if time.time() - last > 600:
            last = time.time()
            for n in procs:
                tail = open(f"/kaggle/working/logs/{n}.log").read().strip().splitlines()[-3:]
                print(f"--- {n}: " + " | ".join(tail), flush=True)
    bad = [n for n, p in procs.items() if p.returncode != 0]
    for n in procs:
        print(f"===== {n} (rc={procs[n].returncode}) last lines =====")
        print("\n".join(open(f"/kaggle/working/logs/{n}.log").read().splitlines()[-15:]))
    if bad:
        raise RuntimeError(f"failed: {bad} (see /kaggle/working/logs)")

def run(cmd):
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
os.makedirs("/kaggle/working/logs", exist_ok=True)
NGPU = torch.cuda.device_count()
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(NGPU)])
assert NGPU >= 1, "Activa el acelerador GPU (T4 x2) en Settings"

In [ ]:
run('python run_phase1.py --preflight --device cuda:0 --out /kaggle/working/out --cache /kaggle/working/cache')

In [ ]:
import os
if not os.path.exists('/kaggle/working/out/v1_screening.json'):
    run('python run_phase1.py --screen_only --device cuda:0 --out /kaggle/working/out --cache /kaggle/working/cache')
print(open('/kaggle/working/out/v1_screening.json').read()[:600])

In [ ]:
def seed_cmd(s, gpu):
    r = f' --resume_from {RESUME_DIR}/phase1_seed{s}.json' if RESUME_DIR else ''
    return f'python run_phase1.py --seed {s} --device cuda:{gpu} --lmeval_limit {LMEVAL_LIMIT} --out /kaggle/working/out --cache /kaggle/working/cache' + r
if NGPU >= 2:
    launch([(f'seed{s}', seed_cmd(s, i)) for i, s in enumerate(SEEDS)])
else:
    launch([('seeds', ' && '.join(seed_cmd(s, 0) for s in SEEDS))])

In [ ]:
run('python aggregate.py --inp /kaggle/working/out --out /kaggle/working/tables')